# 01. Logits, logprobs, softmax, and sigmoid

This notebook introduces raw model outputs, normalization, and the relationship between probabilities and log probabilities.

## Key ideas

- **Logits** are raw model scores before normalization.
- **Softmax** turns a vector of logits into a probability distribution across classes.
- **Sigmoid** turns a single logit into a binary probability.
- **Log probability** is the logarithm of a probability, usually natural log.
- You can recover probability from logprob with `exp(logprob)`.

In [ ]:

import math
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "sshleifer/tiny-gpt2"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()


In [ ]:

logits_example = torch.tensor([2.0, 1.0, 0.1])
probs_example = torch.softmax(logits_example, dim=0)
logprobs_example = torch.log(probs_example)

print('logits:', logits_example)
print('probabilities:', probs_example)
print('log probabilities:', logprobs_example)
print('recover probabilities with exp(logprob):', torch.exp(logprobs_example))


In [ ]:

binary_logits = torch.linspace(-6, 6, 200)
sigmoid_probs = torch.sigmoid(binary_logits)

plt.figure(figsize=(6,4))
plt.plot(binary_logits.numpy(), sigmoid_probs.numpy())
plt.title('Sigmoid maps one logit to a binary probability')
plt.xlabel('logit')
plt.ylabel('sigmoid(logit)')
plt.grid(True)
plt.show()


## Token-level example from a small causal language model

We will inspect the next-token distribution after a short prompt.

In [ ]:

prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

next_token_logits = outputs.logits[0, -1]
next_token_probs = torch.softmax(next_token_logits, dim=0)
next_token_logprobs = torch.log_softmax(next_token_logits, dim=0)

topk = torch.topk(next_token_probs, k=10)
rows = []
for prob, idx in zip(topk.values, topk.indices):
    token = tokenizer.decode([idx.item()])
    logprob = next_token_logprobs[idx].item()
    rows.append((token, idx.item(), next_token_logits[idx].item(), prob.item(), logprob, math.exp(logprob)))

for row in rows:
    print(row)


In [ ]:

tokens = [r[0].replace('
', '\n') for r in rows]
probs = [r[3] for r in rows]
logprobs = [r[4] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].bar(tokens, probs)
axes[0].set_title('Top token probabilities')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(tokens, logprobs)
axes[1].set_title('Top token log probabilities')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
